# 1. Configuração

In [ ]:
from pathlib import Path

import numpy as np
import cv2 # Processamento de imagens
import matplotlib.pyplot as plt
import pandas as pd

# Métrica para comparação dos rostos
from sklearn.metrics.pairwise import cosine_similarity
# Modelo pré-treinado para gerar embeddings faciais
from deepface import DeepFace

In [ ]:
# Configurações do projeto
MODEL_NAME = "Facenet512"

print("Ambiente configurado com sucesso!")
print(f"Modelo facial selecionado: {MODEL_NAME}")

# 2. Carregar dataset

In [ ]:
DATASET_DIR = Path("dataset")

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"O diretório '{DATASET_DIR}' não foi encontrado."
    )

print(f"Dataset encontrado em: {DATASET_DIR.resolve()}")

In [ ]:
# Descobrindo as pessoas automáticamente
pessoas = sorted([
    pasta.name
    for pasta in DATASET_DIR.iterdir()
    if pasta.is_dir()
])

print(f"Quantidade de pessoas encontradas: {len(pessoas)}")

for pessoa in pessoas:
    print(f"- {pessoa}")

In [ ]:
# Separando teste e treino
dataset = {}

for pessoa in pessoas:
    pasta = DATASET_DIR / pessoa

    # Imagens 1, 2, 3 e 4 = treinamento
    imagens_treino = []

    for numero in range(1, 5):
        imagens = list(pasta.glob(f"{numero}_*"))

        if imagens:
            imagens_treino.append(imagens[0])

    # Imagem 5 = teste
    imagens_teste = list(pasta.glob("5_*"))
    imagem_teste = imagens_teste[0] if imagens_teste else None

    dataset[pessoa] = {
        "treino": imagens_treino,
        "teste": imagem_teste
    }

Validação

In [ ]:
dataset_valido = True

for pessoa, dados in dataset.items():

    print(f"\nPessoa: {pessoa}")

    print("Treino:")
    for imagem in dados["treino"]:
        print(f"  - {imagem.name}")

    print("Teste:")
    if dados["teste"]:
        print(f"  - {dados['teste'].name}")
    else:
        print("  - NÃO ENCONTRADA")

In [ ]:
if dataset_valido:
    print("\nDataset validado com sucesso!")
else:
    print("\nExistem problemas na organização do dataset.")

# 3. Detecção facial

In [ ]:
DETECTOR_BACKEND = "opencv"

print(f"Detector facial selecionado: {DETECTOR_BACKEND}")

In [ ]:
def detectar_rosto(caminho_imagem):

    # Carrega a imagem original
    imagem = cv2.imread(str(caminho_imagem))

    if imagem is None:
        raise ValueError(
            f"Não foi possível carregar a imagem: {caminho_imagem}"
        )

    # Detecta as faces presentes na imagem
    faces = DeepFace.extract_faces(
        img_path=str(caminho_imagem),
        detector_backend=DETECTOR_BACKEND,
        enforce_detection=True,
        align=True
    )

    # Para nosso dataset esperamos exatamente um rosto
    if len(faces) != 1:
        raise ValueError(
            f"Esperava exatamente 1 rosto em '{caminho_imagem.name}', "
            f"mas foram encontrados {len(faces)}."
        )

    face = faces[0]

    # Coordenadas retornadas pelo detector
    area = face["facial_area"]

    x = area["x"]
    y = area["y"]
    w = area["w"]
    h = area["h"]

    # Recorta o rosto diretamente da imagem original
    rosto_recortado = imagem[y:y+h, x:x+w]

    return {
        "imagem_original": imagem,
        "bounding_box": (x, y, w, h),
        "rosto": rosto_recortado
    }

explicar bounding box

In [ ]:
pessoa_teste = list(dataset.keys())[0]
imagem_teste = dataset[pessoa_teste]["treino"][0]

resultado = detectar_rosto(imagem_teste)

print(f"Pessoa: {pessoa_teste}")
print(f"Imagem: {imagem_teste.name}")
print(f"Bounding box: {resultado['bounding_box']}")

In [ ]:
def desenhar_bounding_box(imagem, bounding_box):

    # Trabalhamos sobre uma cópia para não alterar a imagem original
    imagem_box = imagem.copy()

    x, y, w, h = bounding_box

    cv2.rectangle(
        imagem_box,
        (x, y),
        (x + w, y + h),
        (0, 255, 0),
        3
    )

    return imagem_box

In [ ]:
imagem_box = desenhar_bounding_box(
    resultado["imagem_original"],
    resultado["bounding_box"]
)

Visualizando as 3 etapas juntas

In [ ]:
def visualizar_deteccao(resultado, titulo="Detecção facial"):

    original = resultado["imagem_original"]

    com_box = desenhar_bounding_box(
        original,
        resultado["bounding_box"]
    )

    rosto = resultado["rosto"]

    # OpenCV trabalha em BGR.
    # Matplotlib espera RGB.
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    box_rgb = cv2.cvtColor(com_box, cv2.COLOR_BGR2RGB)
    rosto_rgb = cv2.cvtColor(rosto, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 4))

    # Imagem original
    plt.subplot(1, 3, 1)
    plt.imshow(original_rgb)
    plt.title("Imagem original")
    plt.axis("off")

    # Bounding box
    plt.subplot(1, 3, 2)
    plt.imshow(box_rgb)
    plt.title("Face detectada")
    plt.axis("off")

    # Rosto recortado
    plt.subplot(1, 3, 3)
    plt.imshow(rosto_rgb)
    plt.title("Rosto recortado")
    plt.axis("off")

    plt.suptitle(titulo)
    plt.tight_layout()
    plt.show()

In [ ]:
visualizar_deteccao(
    resultado,
    titulo=f"{pessoa_teste} - {imagem_teste.name}"
)

Executar para todas as imagens

In [ ]:
resultados_deteccao = {}

for pessoa, dados in dataset.items():

    resultados_deteccao[pessoa] = {
        "treino": [],
        "teste": None
    }

    print(f"\nProcessando: {pessoa}")

    # Imagens de treinamento
    for caminho in dados["treino"]:

        try:
            resultado = detectar_rosto(caminho)

            resultados_deteccao[pessoa]["treino"].append({
                "arquivo": caminho,
                "resultado": resultado
            })

            print(f"  OK   | {caminho.name}")

        except Exception as erro:
            print(f"  ERRO | {caminho.name} | {erro}")

    # Imagem de teste
    try:
        caminho = dados["teste"]

        resultado = detectar_rosto(caminho)

        resultados_deteccao[pessoa]["teste"] = {
            "arquivo": caminho,
            "resultado": resultado
        }

        print(f"  OK   | {caminho.name} [TESTE]")

    except Exception as erro:
        print(f"  ERRO | {caminho.name} | {erro}")

In [ ]:
# Mostrando uma detecção de teste para cada pessoa
for pessoa, dados in resultados_deteccao.items():

    if dados["treino"]:

        exemplo = dados["treino"][0]

        visualizar_deteccao(
            exemplo["resultado"],
            titulo=f"{pessoa} - {exemplo['arquivo'].name}"
        )

# 4. Extração dos embeddings

# 5. Reconhecimento facial

# 6. Caso "Desconhecido"

# 7. Avaliação do modelo